# Vincio Quickstart — the five-minute tour

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ohswedd/vincio/blob/main/examples/notebooks/01_quickstart.ipynb)

**Vincio** is a Python context-engineering platform. The single entry point is `from vincio import ContextApp`.

This notebook is the first thing to read. It walks the whole core loop in four short steps:

1. A plain `app.run()` — and the `trace_id` + `cost_usd` stamped on every result.
2. **Typed output** — get a validated Pydantic object back, not a string.
3. **Grounded QA** — attach your own documents and answer *only* from them, with citations.
4. A short **multi-turn chat**.

Everything runs **fully offline** on the deterministic mock provider — no API keys, no network.

> **To run against a real model**, set `VINCIO_PROVIDER` (e.g. `openai`) and the matching API key (e.g. `OPENAI_API_KEY`), optionally `VINCIO_MODEL`. The same code then calls a real model.

In [ ]:
!pip install -q vincio

## The provider helper

A bare `ContextApp()` defaults to OpenAI and needs an API key. To stay offline-first, every cell builds its app through this tiny helper: it returns the bundled **mock provider** by default, and a **real provider** the moment you set `VINCIO_PROVIDER`. Run this cell once; the rest of the notebook uses it.

In [ ]:
import os

from vincio.providers import MockProvider, build_provider


def _provider():
    """Offline mock by default; a real provider when VINCIO_PROVIDER is set."""
    name = os.environ.get("VINCIO_PROVIDER", "mock")
    if name == "mock":
        return MockProvider(), "mock-1"
    return build_provider(name), os.environ.get("VINCIO_MODEL", "gpt-4o-mini")

from vincio import ContextApp

print("Vincio ready — provider:", os.environ.get("VINCIO_PROVIDER", "mock"))

## 1. Hello, ContextApp

A `ContextApp` wraps a provider + model and runs a single governed turn. Every call returns a `RunResult` carrying not just the `output`, but observability stamped in for free: a `trace_id` (to correlate logs/audit) and a `cost_usd` (token spend).

In [ ]:
provider, model = _provider()
app = ContextApp(name="hello", provider=provider, model=model)

result = app.run("Say hello in one sentence.")
print("output   :", result.output)
print("trace_id :", result.trace_id)
print(f"cost_usd : ${result.cost_usd:.6f}")
print("tokens   :", result.usage.total_tokens, "total")

## 2. Typed output

Pass `output_schema=` and the app parses and validates the model's reply into that Pydantic type. `result.output` is then a typed object — attribute access, IDE completion, and a validation error if the model strays off-schema.

Offline, the mock provider auto-generates a schema-valid instance, so this is fully deterministic. With a real provider the schema is sent to the model and the same parsing/validation applies.

In [ ]:
from pydantic import BaseModel, Field


class TicketClassification(BaseModel):
    """The structured shape we want every classification to conform to."""
    label: str = Field(description="bug | billing | feature | other")
    confidence: float
    reason: str

provider, model = _provider()
app = ContextApp(
    name="support_triage",
    output_schema=TicketClassification,
    provider=provider,
    model=model,
)

result = app.run("I was charged twice this month.")
out = result.output  # a validated TicketClassification instance
print("type      :", type(out).__name__)
print("label     :", out.label)
print("confidence:", out.confidence)
print("reason    :", out.reason)

## 3. Grounded QA with citations

Attach your own documents as a source and the app retrieves, grounds, and cites. We write two small markdown docs to a temp dir, add them with **hybrid** (keyword + vector) retrieval, then turn on the `answer_only_from_sources` policy so the model may only use retrieved evidence. `result.citations` reports exactly which chunks backed the answer — the core of trustworthy RAG.

*(Offline, the mock returns placeholder answer text but real citations and cost — enough to see the grounding pipeline work.)*

In [ ]:
import tempfile
from pathlib import Path

# Lay down two tiny docs in a temp folder (nothing touches your project tree).
docs_dir = Path(tempfile.mkdtemp()) / "docs"
docs_dir.mkdir(parents=True, exist_ok=True)
(docs_dir / "refund_policy.md").write_text(
    "# Refund Policy\n\n"
    "Customers on the Pro plan may request refunds within 30 days with no fee.\n\n"
    "| Plan | Window | Fee |\n|---|---|---|\n| Pro | 30 days | $0 |\n| Basic | 14 days | $5 |\n",
    encoding="utf-8",
)
(docs_dir / "terms.md").write_text(
    "# Terms of Service\n\n"
    "The subscription renews automatically unless terminated 60 days before renewal. "
    "The initial term is 24 months.\n",
    encoding="utf-8",
)

provider, model = _provider()
app = ContextApp(name="docs_qa", provider=provider, model=model)

# Index the folder: adaptive chunking + hybrid retrieval are sensible defaults.
app.add_source("docs", path=str(docs_dir), chunking="adaptive", retrieval="hybrid")

# Guardrail: only answer from retrieved evidence.
app.set_policy("answer_only_from_sources", True)

result = app.run("What is the refund window for the Pro plan?")
print("answer   :", result.output)
print("citations:", result.citations)
print("trace_id :", result.trace_id)

## 4. A short multi-turn chat

`app.assistant(...)` is a session-aware chat layer: each `send` is a full governed run, but turns are threaded under one session with memory, so later turns can reference earlier context. A chat product is a short loop, not hand-wired plumbing.

In [ ]:
provider, model = _provider()
app = ContextApp(name="support_chat", provider=provider, model=model)
chat = app.assistant(user_id="cust-42")

for user_msg in [
    "What's my refund window? My plan is Pro.",
    "Thanks, that's all I needed.",
]:
    turn = chat.send(user_msg)
    print(f"user      : {user_msg}")
    print(f"assistant : {turn.text}\n")

# The session retains the whole exchange (user + assistant turns).
print(f"session {chat.session_id}: {len(chat.history())} messages retained")

## Where to next

You have seen the whole core loop: a governed run with built-in cost/trace, typed Pydantic output, grounded RAG with citations, and a multi-turn chat.

Continue with the runnable examples in the repo:

- `examples/02_retrieval_rag.py` — hybrid fusion, query understanding, GraphRAG, multimodal evidence.
- `examples/04_agents_and_tools.py` — tools, approvals, and write-gating.
- `examples/06_structured_output.py` — constrained decoding, streaming validation, self-correction.
- `examples/07_evaluation_observability.py` — datasets, metrics, and gates.
- `examples/15_governed_text_to_query.py`, `16_data_analysis_agent.py`, `17_charts_cited_artifacts.py` — the data plane.

Every example runs offline on the mock provider and against a real model with one env var. Happy building!